## Changing the data from .csv.zst to .delta

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import date_trunc, col

# set schema
schema = StructType([
    StructField("mmsi", LongType(), True),
    StructField("base_date_time", TimestampType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("sog", DoubleType(), True),
    StructField("cog", DoubleType(), True),
    StructField("heading", DoubleType(), True),
    StructField("vessel_name", StringType(), True),
    StructField("imo", StringType(), True),
    StructField("call_sign", StringType(), True),
    StructField("vessel_type", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("length", DoubleType(), True),
    StructField("width", DoubleType(), True),
    StructField("draft", DoubleType(), True),
    StructField("cargo", IntegerType(), True),
    StructField("transceiver", StringType(), True),
])
# read .csv.zst
df = spark.read.option("header", "true")\
    .schema(schema)\
    .csv('/Volumes/workspace/default/raw_data/ais-*.csv.zst')
# create month column for partitions
df = df.withColumn("month", date_trunc("month", "base_date_time").cast("date"))

# write to delta
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("month") \
    .save('/Volumes/workspace/default/raw_data/ais_bronze_delta')

## Comparing read time for .csv.zst to delta

In [0]:
import time

# CSV
start = time.time()
df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "/Volumes/workspace/default/raw_data/ais-*.csv.zst"
)
df_csv.count()
print(f"CSV: {time.time() - start:.1f}s")

# Delta
start = time.time()
df_bronze = spark.read.format("delta").load("/Volumes/workspace/default/raw_data/ais_bronze_delta")
df_bronze.count()
print(f"Delta: {time.time() - start:.1f}s")

CSV: 1703.2s
Delta: 2.1s


## Get Count of Bronze Rows

In [0]:
print(df_delta.count())

3021153531


## Creating Silver Layer

- Add in vessel_type
- Do temporal thinning

Not sure show much either of these will matter now that data is in delta format, but I do this when working with python...

In [0]:
#imported 2nd time, didn't want to rerun first code block
from pyspark.sql.functions import date_trunc, row_number, col, broadcast 
from pyspark.sql import Window

# Read bronze
df = spark.read.format("delta").load("/Volumes/workspace/default/raw_data/ais_bronze_delta")

# Temporal thinning - one point per hour per vessel
window = Window.partitionBy("mmsi", date_trunc("hour", "base_date_time")).orderBy("base_date_time")
df = df.withColumn("rank", row_number().over(window)).filter(col("rank") == 1).drop("rank")

# Ship type lookup (googled AIS ship type, got a dictionary from google's gemini search feature)
ship_type_dict = {
    0: "Not available (default)",
    1: "Reserved for future use",
    2: "Reserved for future use",
    3: "Reserved for future use",
    4: "Reserved for future use",
    5: "Reserved for future use",
    6: "Reserved for future use",
    7: "Reserved for future use",
    8: "Reserved for future use",
    9: "Reserved for future use",
    10: "Reserved for future use",
    11: "Reserved for future use",
    12: "Reserved for future use",
    13: "Reserved for future use",
    14: "Reserved for future use",
    15: "Reserved for future use",
    16: "Reserved for future use",
    17: "Reserved for future use",
    18: "Reserved for future use",
    19: "Reserved for future use",
    20: "Wing in ground (WIG), all ships of this type",
    21: "Wing in ground (WIG), Hazardous category A",
    22: "Wing in ground (WIG), Hazardous category B",
    23: "Wing in ground (WIG), Hazardous category C",
    24: "Wing in ground (WIG), Hazardous category D",
    25: "Wing in ground (WIG), Reserved for future use",
    26: "Wing in ground (WIG), Reserved for future use",
    27: "Wing in ground (WIG), Reserved for future use",
    28: "Wing in ground (WIG), Reserved for future use",
    29: "Wing in ground (WIG), Reserved for future use",
    30: "Fishing",
    31: "Towing",
    32: "Towing: length exceeds 200m or breadth exceeds 25m",
    33: "Dredging or underwater ops",
    34: "Diving ops",
    35: "Military ops",
    36: "Sailing",
    37: "Pleasure Craft",
    38: "Reserved",
    39: "Reserved",
    40: "High speed craft (HSC), all ships of this type",
    41: "High speed craft (HSC), Hazardous category A",
    42: "High speed craft (HSC), Hazardous category B",
    43: "High speed craft (HSC), Hazardous category C",
    44: "High speed craft (HSC), Hazardous category D",
    45: "High speed craft (HSC), Reserved for future use",
    46: "High speed craft (HSC), Reserved for future use",
    47: "High speed craft (HSC), Reserved for future use",
    48: "High speed craft (HSC), Reserved for future use",
    49: "High speed craft (HSC), No additional information",
    50: "Pilot Vessel",
    51: "Search and Rescue vessel",
    52: "Tug",
    53: "Port Tender",
    54: "Anti-pollution equipment",
    55: "Law Enforcement",
    56: "Spare - Local Vessel",
    57: "Spare - Local Vessel",
    58: "Medical Transport",
    59: "Noncombatant ship according to RR Resolution No. 18",
    60: "Passenger, all ships of this type",
    61: "Passenger, Hazardous category A",
    62: "Passenger, Hazardous category B",
    63: "Passenger, Hazardous category C",
    64: "Passenger, Hazardous category D",
    65: "Passenger, Reserved for future use",
    66: "Passenger, Reserved for future use",
    67: "Passenger, Reserved for future use",
    68: "Passenger, Reserved for future use",
    69: "Passenger, No additional information",
    70: "Cargo, all ships of this type",
    71: "Cargo, Hazardous category A",
    72: "Cargo, Hazardous category B",
    73: "Cargo, Hazardous category C",
    74: "Cargo, Hazardous category D",
    75: "Cargo, Reserved for future use",
    76: "Cargo, Reserved for future use",
    77: "Cargo, Reserved for future use",
    78: "Cargo, Reserved for future use",
    79: "Cargo, No additional information",
    80: "Tanker, all ships of this type",
    81: "Tanker, Hazardous category A",
    82: "Tanker, Hazardous category B",
    83: "Tanker, Hazardous category C",
    84: "Tanker, Hazardous category D",
    85: "Tanker, Reserved for future use",
    86: "Tanker, Reserved for future use",
    87: "Tanker, Reserved for future use",
    88: "Tanker, Reserved for future use",
    89: "Tanker, No additional information",
    90: "Other Type, all ships of this type",
    91: "Other Type, Hazardous category A",
    92: "Other Type, Hazardous category B",
    93: "Other Type, Hazardous category C",
    94: "Other Type, Hazardous category D",
    95: "Other Type, Reserved for future use",
    96: "Other Type, Reserved for future use",
    97: "Other Type, Reserved for future use",
    98: "Other Type, Reserved for future use",
    99: "Other Type, no additional information",
}

df_ship_types = spark.createDataFrame(
    [(k, v) for k, v in ship_type_dict.items()],
    ["vessel_type", "vessel_type_name"]
)

df = df.join(broadcast(df_ship_types), on="vessel_type", how="left")

# Write silver
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("month") \
    .save("/Volumes/workspace/default/raw_data/ais_silver_delta")

# Read Silver and print schema
df_silver = spark.read.format("delta").load('/Volumes/workspace/default/raw_data/ais_silver_delta')
df_silver.printSchema()

root
 |-- vessel_type: integer (nullable = true)
 |-- mmsi: long (nullable = true)
 |-- base_date_time: timestamp (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- sog: double (nullable = true)
 |-- cog: double (nullable = true)
 |-- heading: double (nullable = true)
 |-- vessel_name: string (nullable = true)
 |-- imo: string (nullable = true)
 |-- call_sign: string (nullable = true)
 |-- status: string (nullable = true)
 |-- length: double (nullable = true)
 |-- width: double (nullable = true)
 |-- draft: double (nullable = true)
 |-- cargo: integer (nullable = true)
 |-- transceiver: string (nullable = true)
 |-- month: date (nullable = true)
 |-- vessel_type_name: string (nullable = true)



## Read Silver and Get Count of Rows

In [0]:
df_silver = spark.read.format("delta").load("/Volumes/workspace/default/raw_data/ais_silver_delta")
print(df_silver.count())

133710106
